# 04 — Baseline Model: ACLED Only

This notebook estimates the **baseline negative binomial regression model** using only ACLED conflict-history variables. It serves as the benchmark against which the extended model (ACLED + GDELT) is compared in notebook 05.

| | |
|---|---|
| Data source | ACLED only |
| Predictors | `events_lag1–4`, `events_ma4`, `events_ma8`, `events_std4`, `conflict_weeks_last8` |
| Model | Negative binomial regression |
| Outcome | Count of political violence events h weeks ahead |
| Horizons | 1, 2, 4 weeks |
| Train period | 2015–2021 |
| Test period | 2022–2023 |
| Helper functions | Imported from `model_helpers.py` |

---

## Why negative binomial regression?

The outcome variable is a weekly event count. Poisson regression, the standard model for count data, assumes that the variance equals the mean. A preliminary check on the ACLED data found a variance-to-mean ratio of approximately 25 across all five countries, far exceeding the Poisson assumption. Negative binomial regression adds a dispersion parameter (alpha) that explicitly models this extra variance. A statistically significant, positive alpha in the results confirms the choice.

## Why these predictors?

A full candidate set of 14 ACLED-derived variables was evaluated. These span four conceptual groups:

- **Short-term lags** (`events_lag1–4`): event counts from 1 to 4 weeks before the prediction point, capturing immediate conflict history
- **Moving averages** (`events_ma4`, `events_ma8`): 4- and 8-week rolling means, capturing the medium-term conflict regime
- **Volatility** (`events_std4`): rolling standard deviation, capturing erratic conflict dynamics
- **Persistence** (`conflict_weeks_last8`): count of weeks with any conflict in the past 8 weeks, capturing sustained conflict streaks

Including all 14 candidates simultaneously produces extreme multicollinearity (VIF > 900) because lags and moving averages are all derived from the same event count series. The chosen set retains one or two representatives from each conceptual group. Although high VIF remains among the lag and moving average terms, this is acceptable when the goal is predictive accuracy rather than causal interpretation of individual coefficients, and when the model converges and performs well on held-out data, both of which hold here.

## Why a temporal train/test split?

The data are organised as a time series. Random cross-validation would leak future information into training through the lagged predictors: a row that is used for validation might have its lag values drawn from rows used for training that come after it in time. A strict temporal split (train 2015–2021, test 2022–2023) avoids this and mirrors the real-world use case of predicting future conflict from past data.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings
import sys
import matplotlib.pyplot as plt
from pathlib import Path

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────────
# Paths relative to repo root
REPO_ROOT = Path().resolve().parent  # one level up from notebooks/

PROC_DIR  = REPO_ROOT / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = REPO_ROOT / "Results" / "baseline"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Import shared helpers ──────────────────────────────────────────────────────
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from model_helpers import (
    HORIZONS, TRAIN_END, TEST_START,
    BASELINE_PREDICTORS,
    fix_float_col, get_split,
    fit_negbinom, predict_negbinom,
    evaluate, check_vif, build_coef_table,
)

print(f"Baseline predictors ({len(BASELINE_PREDICTORS)}): {BASELINE_PREDICTORS}")
print(f"Train: up to {TRAIN_END.date()}")
print(f"Test:  from  {TEST_START.date()}")

## 1. Load Data

In [ ]:
# Load the ACLED-only panel produced by notebook 01
panel = pd.read_csv(PROC_DIR / "acled_country_week.csv", parse_dates=["week_start"])

# Repair any multi-dot float artefacts that may have survived the CSV round-trip
non_numeric = ["COUNTRY", "week_start"]
for col in panel.columns:
    if col not in non_numeric:
        panel[col] = fix_float_col(panel[col])

print(f"Panel shape: {panel.shape}")
print(f"Countries:   {sorted(panel['COUNTRY'].unique())}")
print(f"Date range:  {panel['week_start'].min().date()} → {panel['week_start'].max().date()}")

print("\nPredictor availability:")
for p in BASELINE_PREDICTORS:
    present = p in panel.columns
    nan_ct  = panel[p].isna().sum() if present else "N/A"
    print(f"  {'✓' if present else '✗'} {p}  (NaN: {nan_ct})")

## 2. Overdispersion Check

Confirms that the outcome variable is overdispersed, justifying the choice of negative binomial over Poisson regression. A variance-to-mean ratio substantially above 1 indicates overdispersion.

In [ ]:
print("Overdispersion check on n_events (all country-weeks):")
mean_val = panel["n_events"].mean()
var_val  = panel["n_events"].var()
print(f"  Mean:     {mean_val:.3f}")
print(f"  Variance: {var_val:.3f}")
print(f"  Ratio:    {var_val/mean_val:.2f}  (>>1 confirms NB is appropriate over Poisson)")

print("\nPer-country:")
print(
    panel.groupby("COUNTRY")["n_events"]
    .agg(mean="mean", var="var")
    .assign(ratio=lambda d: d["var"] / d["mean"])
    .round(2)
)

## 3. Multicollinearity Check (VIF)

VIF is computed for the baseline predictor set before fitting. Values above 10 indicate potentially problematic multicollinearity. As discussed in the thesis methods section, high VIF among the lag and moving average terms is expected because they are all derived from the same event count series. The supervisor confirmed this is acceptable given the predictive goal of the study.

In [ ]:
print("VIF — Baseline predictor set (h=1 training data):")
X_vif, _, _, _, _ = get_split(panel, 1, BASELINE_PREDICTORS)
vif_df = check_vif(X_vif)
print(vif_df.round(2).to_string())

bad = vif_df[vif_df["VIF"] > 10]
if len(bad) > 0:
    print(f"\nNote: VIF > 10 for {bad.index.tolist()}.")
    print("This is expected among lag/MA terms derived from the same series.")
    print("The model converges and performs well — goal of model is overall performance, not individual predictor significance.")
else:
    print("\nAll VIFs below 10 ✓")

## 4. Model Training

One model is estimated per forecast horizon (h = 1, 2, 4 weeks). The predictor set is identical across all three horizons, only the outcome variable changes. This is standard in multi-horizon forecasting: all predictors are lagged relative to the current week and are therefore available at prediction time regardless of how far ahead we are predicting.

Key convergence diagnostics to check:
- `Converged: True` — the optimiser found a solution
- `alpha > 0` and statistically significant; confirms overdispersion
- AIC should increase slightly with horizon as the prediction task becomes harder

In [ ]:
all_results = {}

for h in HORIZONS:
    print(f"\n{'='*55}")
    print(f"HORIZON: {h} week(s) ahead")
    print("="*55)

    X_train, X_test, y_train, y_test, preds = get_split(panel, h, BASELINE_PREDICTORS)

    print("\n  Fitting negative binomial...")
    result = fit_negbinom(X_train, y_train)

    alpha = result.params.get("alpha", np.nan)
    print(f"  Converged:          {result.mle_retvals.get('converged', '?')}")
    print(f"  Log-likelihood:     {result.llf:.2f}")
    print(f"  AIC:                {result.aic:.2f}")
    print(f"  BIC:                {result.bic:.2f}")
    print(f"  Dispersion (alpha): {alpha:.4f}")

    y_pred_train = predict_negbinom(result, X_train)
    y_pred_test  = predict_negbinom(result, X_test)

    train_m = evaluate(y_train, y_pred_train)
    test_m  = evaluate(y_test,  y_pred_test)

    print(f"  Train — RMSE: {train_m['rmse']:.3f}, MAE: {train_m['mae']:.3f}")
    print(f"  Test  — RMSE: {test_m['rmse']:.3f},  MAE: {test_m['mae']:.3f}")

    all_results[h] = dict(
        result=result, X_train=X_train, X_test=X_test,
        y_train=y_train, y_test=y_test,
        y_pred_train=y_pred_train, y_pred_test=y_pred_test,
        train_metrics=train_m, test_metrics=test_m, predictors=preds,
    )

## 5. Performance Summary

In [ ]:
print("BASELINE MODEL — PERFORMANCE SUMMARY (test set)")
print("=" * 58)
print(f"{'Horizon':<12} {'Train RMSE':>11} {'Train MAE':>10} {'Test RMSE':>10} {'Test MAE':>10}")
print("-" * 58)
for h in HORIZONS:
    tm = all_results[h]["train_metrics"]
    vm = all_results[h]["test_metrics"]
    print(f"h={h} week{'s' if h>1 else '':<7} "
          f"{tm['rmse']:>11.3f} {tm['mae']:>10.3f} "
          f"{vm['rmse']:>10.3f} {vm['mae']:>10.3f}")

# Save for comparison with extended model in notebook 05
rows = []
for h in HORIZONS:
    for split, key in [("train", "train_metrics"), ("test", "test_metrics")]:
        row = {"horizon": h, "split": split}
        row.update(all_results[h][key])
        rows.append(row)
pd.DataFrame(rows).to_csv(RESULTS_DIR / "baseline_metrics.csv", index=False)
print(f"\nMetrics saved → {RESULTS_DIR / 'baseline_metrics.csv'}")

## 6. Coefficient Table (h = 1)

Coefficients are on the log scale (standard for negative binomial with a log link). They are estimated on standardised predictors, so magnitudes are directly comparable: a coefficient of 0.3 has three times the per-standard-deviation effect of a coefficient of 0.1.

The dispersion parameter alpha is reported separately — it is not a predictor coefficient but a model fit statistic. A significant positive alpha confirms overdispersion.

In [ ]:
res_h1    = all_results[1]["result"]
coef_df   = build_coef_table(res_h1)
alpha_val = res_h1.params.get("alpha", np.nan)
alpha_se  = res_h1.bse.get("alpha",   np.nan)

print("Coefficient Table — Baseline Model, h=1 (standardised predictors):")
print(coef_df.to_string())
print(f"\nDispersion parameter alpha: {alpha_val:.4f} (SE={alpha_se:.4f})")
print("Alpha > 0 and significant confirms overdispersion; NB preferred over Poisson.")

coef_df.to_csv(RESULTS_DIR / "baseline_coefs_h1.csv")
print(f"\nSaved → {RESULTS_DIR / 'baseline_coefs_h1.csv'}")

## 7. Visualisations

### 7a. Residual distribution

Residuals = predicted minus observed. Ideally centred on zero. A positive mean indicates the model tends to overpredict; a long right tail indicates occasional very large overpredictions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, h in zip(axes, HORIZONS):
    errors = all_results[h]["y_pred_test"] - np.array(all_results[h]["y_test"])
    ax.hist(errors, bins=40, color="steelblue", edgecolor="white", alpha=0.85)
    ax.axvline(0,             color="red",    lw=1.5, ls="--", label="Zero error")
    ax.axvline(errors.mean(), color="orange", lw=1.2, ls=":",
               label=f"Mean={errors.mean():.2f}")
    ax.set_title(f"h = {h} week{'s' if h>1 else ''}")
    ax.set_xlabel("Predicted − Observed")
    ax.set_ylabel("Frequency")
    ax.legend(fontsize=8)

plt.suptitle("Baseline Model — Residual Distribution (Test Set)", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "baseline_residuals.png", dpi=150, bbox_inches="tight")
plt.show()

### 7b. Time series by country (h = 1)

Shows how well the model tracks weekly conflict over the 2022–2023 test period for each country.

In [ ]:
h             = 1
X_test_h1     = all_results[h]["X_test"]
test_data     = panel.loc[X_test_h1.index].copy()
test_data["y_true"] = all_results[h]["y_test"].values
test_data["y_pred"] = all_results[h]["y_pred_test"]

TARGET_COUNTRIES = sorted(panel["COUNTRY"].unique())
fig, axes = plt.subplots(len(TARGET_COUNTRIES), 1,
                          figsize=(14, 3.5 * len(TARGET_COUNTRIES)), sharex=True)

for ax, country in zip(axes, TARGET_COUNTRIES):
    sub = test_data[test_data["COUNTRY"] == country].sort_values("week_start")
    ax.fill_between(sub["week_start"], sub["y_true"],
                    alpha=0.4, color="steelblue", label="Observed")
    ax.plot(sub["week_start"], sub["y_pred"],
            color="red", lw=1.2, label="Predicted")
    ax.set_title(country, fontsize=10)
    ax.set_ylabel("Events")
    ax.legend(fontsize=8, loc="upper right")

axes[-1].set_xlabel("Week")
plt.suptitle("Baseline Model — Observed vs Predicted by Country (h=1, test period)",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "baseline_timeseries_h1.png", dpi=150, bbox_inches="tight")
plt.show()

### 7c. Coefficient forest plot (h = 1)

In [ ]:
coef_plt = coef_df.sort_values("Coefficient")
fig, ax  = plt.subplots(figsize=(8, 5))
y_pos    = np.arange(len(coef_plt))

ax.barh(
    y_pos, coef_plt["Coefficient"],
    xerr=1.96 * coef_plt["Std. Error"],
    color=["tomato" if c < 0 else "steelblue" for c in coef_plt["Coefficient"]],
    error_kw={"elinewidth": 1.2, "capsize": 3},
    height=0.5, alpha=0.85
)

for i, (_, row) in enumerate(coef_plt.iterrows()):
    sig = ("***" if row["p-value"]<0.001 else "**" if row["p-value"]<0.01
           else "*" if row["p-value"]<0.05 else "(ns)")
    ax.text(coef_plt["Coefficient"].abs().max() * 1.15, i, sig, va="center", fontsize=9)

ax.axvline(0, color="black", lw=0.8, ls="--")
ax.set_yticks(y_pos)
ax.set_yticklabels(coef_plt.index, fontsize=9)
ax.set_xlabel("Coefficient (log scale, standardised predictors)")
ax.set_title("Baseline Model Coefficients — h=1 (95% CI)", fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "baseline_coef_plot_h1.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Save Predictions

Predictions are saved for all horizons so notebook 05 can load them for the baseline-vs-extended comparison without needing to re-run this notebook.

In [ ]:
for h in HORIZONS:
    pd.DataFrame({
        "y_observed":  np.array(all_results[h]["y_test"]),
        "y_predicted": all_results[h]["y_pred_test"],
    }).to_csv(RESULTS_DIR / f"baseline_predictions_h{h}.csv", index=False)
    print(f"Saved baseline predictions h={h} → {RESULTS_DIR / f'baseline_predictions_h{h}.csv'}")